# CT107-3-3 Text Analytics and Sentiment Analysis
## Part A — Q4: Sentence Probabilities using Bigram Language Models
**Dataset:** `Data_3.txt`

### Setup — Import Libraries and Load Data

In [1]:
import re
import math
from collections import defaultdict

DATA_PATH = r"D:\TXSA\Part A Dataset\Part A Dataset\Data_3.txt"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = f.read()

print("Raw file content:")
print(raw)

Raw file content:
Training Corpus
~~~~~~~~~~~~~
<s> He read a book </s>
<s> I read a different book </s>
<s> He read a book by Danielle </s>

Calculate sentence probability for the following sentence
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
<s> I read a different book by Danielle </s>


---
### Pre-processing

The file contains three training sentences delimited by `<s>` and `</s>` sentence boundary markers, followed by a test sentence. We extract and tokenise each sentence, retaining the boundary markers as special tokens.

In [2]:
# Extract all <s> ... </s> sentence blocks
all_sentence_matches = re.findall(r'<s>(.*?)</s>', raw)

# Training: first 3 sentences; Test: last sentence
training_sentences = []
for line in all_sentence_matches[:3]:
    tokens = ['<s>'] + line.strip().split() + ['</s>']
    training_sentences.append(tokens)

test_sentence = ['<s>'] + all_sentence_matches[-1].strip().split() + ['</s>']

print("Training sentences (with sentence pads):")
for i, s in enumerate(training_sentences, 1):
    print(f"  {i}: {s}")

print(f"\nTest sentence:")
print(f"     {test_sentence}")

Training sentences (with sentence pads):
  1: ['<s>', 'He', 'read', 'a', 'book', '</s>']
  2: ['<s>', 'I', 'read', 'a', 'different', 'book', '</s>']
  3: ['<s>', 'He', 'read', 'a', 'book', 'by', 'Danielle', '</s>']

Test sentence:
     ['<s>', 'I', 'read', 'a', 'different', 'book', 'by', 'Danielle', '</s>']


---
### Unigram and Bigram Counts from Training Corpus

In [3]:
unigram_counts = defaultdict(int)
bigram_counts  = defaultdict(int)

for sent in training_sentences:
    for w in sent:
        unigram_counts[w] += 1
    for w1, w2 in zip(sent[:-1], sent[1:]):
        bigram_counts[(w1, w2)] += 1

V = len(unigram_counts)  # vocabulary size

print("Unigram Counts:")
for word, cnt in sorted(unigram_counts.items()):
    print(f"  C({word}) = {cnt}")

print(f"\nBigram Counts:")
for bigram, cnt in sorted(bigram_counts.items()):
    print(f"  C{bigram} = {cnt}")

print(f"\nVocabulary size V = {V}")
print(f"Vocabulary: {sorted(unigram_counts.keys())}")

Unigram Counts:
  C(</s>) = 3
  C(<s>) = 3
  C(Danielle) = 1
  C(He) = 2
  C(I) = 1
  C(a) = 3
  C(book) = 3
  C(by) = 1
  C(different) = 1
  C(read) = 3

Bigram Counts:
  C('<s>', 'He') = 2
  C('<s>', 'I') = 1
  C('Danielle', '</s>') = 1
  C('He', 'read') = 2
  C('I', 'read') = 1
  C('a', 'book') = 2
  C('a', 'different') = 1
  C('book', '</s>') = 2
  C('book', 'by') = 1
  C('by', 'Danielle') = 1
  C('different', 'book') = 1
  C('read', 'a') = 3

Vocabulary size V = 10
Vocabulary: ['</s>', '<s>', 'Danielle', 'He', 'I', 'a', 'book', 'by', 'different', 'read']


---
## Q4(b) — Unsmoothed Bigram Model (Manual Computation)
*(3 marks)*

The **unsmoothed bigram probability** formula:

$$P(w_i \mid w_{i-1}) = \frac{C(w_{i-1},\, w_i)}{C(w_{i-1})}$$

The sentence probability is the product of all conditional bigram probabilities:

$$P(S) = \prod_{i=1}^{n} P(w_i \mid w_{i-1})$$

In [4]:
bigrams_test = list(zip(test_sentence[:-1], test_sentence[1:]))

print("Test sentence bigrams:")
print(bigrams_test)
print()
print("Unsmoothed Bigram Probabilities:")
print(f"{'Bigram':<35} {'Count(w-1,w)':<15} {'Count(w-1)':<12} {'Probability'}")
print("-" * 75)

prob_unsmoothed = 1.0
for (w_prev, w_curr) in bigrams_test:
    num  = bigram_counts[(w_prev, w_curr)]
    den  = unigram_counts[w_prev]
    prob = num / den if den > 0 else 0.0
    label = f"P({w_curr}|{w_prev})"
    print(f"{label:<35} {num:<15} {den:<12} {num}/{den} = {prob:.6f}")
    prob_unsmoothed *= prob

print(f"\nUnsmoothed P(sentence) = {prob_unsmoothed:.10f}")
print(f"                       = 1/27 ≈ {1/27:.10f}")
if prob_unsmoothed > 0:
    print(f"log P(sentence)        = {math.log(prob_unsmoothed):.6f}")

Test sentence bigrams:
[('<s>', 'I'), ('I', 'read'), ('read', 'a'), ('a', 'different'), ('different', 'book'), ('book', 'by'), ('by', 'Danielle'), ('Danielle', '</s>')]

Unsmoothed Bigram Probabilities:
Bigram                              Count(w-1,w)    Count(w-1)   Probability
---------------------------------------------------------------------------
P(I|<s>)                            1               3            1/3 = 0.333333
P(read|I)                           1               1            1/1 = 1.000000
P(a|read)                           3               3            3/3 = 1.000000
P(different|a)                      1               3            1/3 = 0.333333
P(book|different)                   1               1            1/1 = 1.000000
P(by|book)                          1               3            1/3 = 0.333333
P(Danielle|by)                      1               1            1/1 = 1.000000
P(</s>|Danielle)                    1               1            1/1 = 1.000000

Uns

### Manual Working — Unsmoothed Bigram Model

Test sentence: `<s> I read a different book by Danielle </s>`

| Step | Bigram | Numerator | Denominator | Probability |
|------|--------|-----------|-------------|-------------|
| 1 | P(I \| \<s\>) | C(\<s\>, I) = 1 | C(\<s\>) = 3 | **1/3** |
| 2 | P(read \| I) | C(I, read) = 1 | C(I) = 1 | **1/1 = 1** |
| 3 | P(a \| read) | C(read, a) = 3 | C(read) = 3 | **3/3 = 1** |
| 4 | P(different \| a) | C(a, different) = 1 | C(a) = 3 | **1/3** |
| 5 | P(book \| different) | C(different, book) = 1 | C(different) = 1 | **1/1 = 1** |
| 6 | P(by \| book) | C(book, by) = 1 | C(book) = 3 | **1/3** |
| 7 | P(Danielle \| by) | C(by, Danielle) = 1 | C(by) = 1 | **1/1 = 1** |
| 8 | P(\</s\> \| Danielle) | C(Danielle, \</s\>) = 1 | C(Danielle) = 1 | **1/1 = 1** |

$$P(S) = \frac{1}{3} \times 1 \times 1 \times \frac{1}{3} \times 1 \times \frac{1}{3} \times 1 \times 1 = \frac{1}{27} \approx 0.037037$$

---
## Q4(c) — Smoothed Bigram Model — Laplace (Add-1) Smoothing (Manual Computation)
*(3 marks)*

**Laplace (Add-1) smoothing** adds 1 to every bigram count to avoid zero probabilities for unseen bigrams:

$$P^*(w_i \mid w_{i-1}) = \frac{C(w_{i-1},\, w_i) + 1}{C(w_{i-1}) + V}$$

where **V** is the vocabulary size (number of unique tokens including sentence boundary markers).

In [5]:
print(f"Vocabulary size V = {V}\n")
print("Smoothed (Laplace) Bigram Probabilities:")
print(f"{'Bigram':<35} {'Num (C+1)':<12} {'Den (C(w-1)+V)':<18} {'Probability'}")
print("-" * 80)

prob_smoothed = 1.0
for (w_prev, w_curr) in bigrams_test:
    num  = bigram_counts[(w_prev, w_curr)] + 1
    den  = unigram_counts[w_prev] + V
    prob = num / den
    label = f"P*({w_curr}|{w_prev})"
    raw_num = bigram_counts[(w_prev, w_curr)]
    raw_den = unigram_counts[w_prev]
    print(f"{label:<35} ({raw_num}+1)={num:<6} ({raw_den}+{V})={den:<10} {num}/{den} = {prob:.8f}")
    prob_smoothed *= prob

print(f"\nSmoothed P(sentence) = {prob_smoothed:.15f}")
print(f"log P(sentence)      = {math.log(prob_smoothed):.6f}")

Vocabulary size V = 10

Smoothed (Laplace) Bigram Probabilities:
Bigram                              Num (C+1)    Den (C(w-1)+V)     Probability
--------------------------------------------------------------------------------
P*(I|<s>)                           (1+1)=2      (3+10)=13         2/13 = 0.15384615
P*(read|I)                          (1+1)=2      (1+10)=11         2/11 = 0.18181818
P*(a|read)                          (3+1)=4      (3+10)=13         4/13 = 0.30769231
P*(different|a)                     (1+1)=2      (3+10)=13         2/13 = 0.15384615
P*(book|different)                  (1+1)=2      (1+10)=11         2/11 = 0.18181818
P*(by|book)                         (1+1)=2      (3+10)=13         2/13 = 0.15384615
P*(Danielle|by)                     (1+1)=2      (1+10)=11         2/11 = 0.18181818
P*(</s>|Danielle)                   (1+1)=2      (1+10)=11         2/11 = 0.18181818

Smoothed P(sentence) = 0.000001224407021
log P(sentence)      = -13.613054


### Manual Working — Laplace Smoothed Bigram Model

Vocabulary size **V = 10** (tokens: `<s>`, `He`, `I`, `read`, `a`, `book`, `different`, `by`, `Danielle`, `</s>`)

| Step | Bigram | Numerator | Denominator | Probability |
|------|--------|-----------|-------------|-------------|
| 1 | P\*(I \| \<s\>) | 1+1 = 2 | 3+10 = 13 | **2/13** |
| 2 | P\*(read \| I) | 1+1 = 2 | 1+10 = 11 | **2/11** |
| 3 | P\*(a \| read) | 3+1 = 4 | 3+10 = 13 | **4/13** |
| 4 | P\*(different \| a) | 1+1 = 2 | 3+10 = 13 | **2/13** |
| 5 | P\*(book \| different) | 1+1 = 2 | 1+10 = 11 | **2/11** |
| 6 | P\*(by \| book) | 1+1 = 2 | 3+10 = 13 | **2/13** |
| 7 | P\*(Danielle \| by) | 1+1 = 2 | 1+10 = 11 | **2/11** |
| 8 | P\*(\</s\> \| Danielle) | 1+1 = 2 | 1+10 = 11 | **2/11** |

$$P^*(S) = \frac{2}{13} \times \frac{2}{11} \times \frac{4}{13} \times \frac{2}{13} \times \frac{2}{11} \times \frac{2}{13} \times \frac{2}{11} \times \frac{2}{11}$$

$$= \frac{2 \times 2 \times 4 \times 2 \times 2 \times 2 \times 2 \times 2}{13^4 \times 11^4} = \frac{512}{28561 \times 14641} = \frac{512}{418{,}195{,}201} \approx 1.224 \times 10^{-6}$$

---
## Q4(d) — Python Implementation: Unsmoothed vs Smoothed Bigram Models
*(4 marks)*

In [6]:
class BigramLanguageModel:
    """
    Bigram language model supporting both unsmoothed and
    Laplace (Add-1) smoothed probability estimation.
    """

    def __init__(self, corpus_sentences, smoothing=False):
        """
        Parameters
        ----------
        corpus_sentences : list of list of str
            Tokenised training sentences (including <s> and </s> markers).
        smoothing : bool
            If True, apply Laplace (Add-1) smoothing.
        """
        self.smoothing = smoothing
        self.unigram   = defaultdict(int)
        self.bigram    = defaultdict(int)
        self._train(corpus_sentences)
        self.V = len(self.unigram)  # vocabulary size

    def _train(self, sentences):
        """Build unigram and bigram count tables from training sentences."""
        for sent in sentences:
            for w in sent:
                self.unigram[w] += 1
            for w1, w2 in zip(sent[:-1], sent[1:]):
                self.bigram[(w1, w2)] += 1

    def word_probability(self, w_prev, w_curr):
        """Return P(w_curr | w_prev) with or without smoothing."""
        if self.smoothing:
            return (self.bigram[(w_prev, w_curr)] + 1) / (self.unigram[w_prev] + self.V)
        else:
            if self.unigram[w_prev] == 0:
                return 0.0
            return self.bigram[(w_prev, w_curr)] / self.unigram[w_prev]

    def sentence_probability(self, sentence_tokens):
        """Return the probability of a sentence as a product of bigram probs."""
        prob = 1.0
        for w_prev, w_curr in zip(sentence_tokens[:-1], sentence_tokens[1:]):
            prob *= self.word_probability(w_prev, w_curr)
        return prob

    def sentence_log_probability(self, sentence_tokens):
        """Return log-probability of a sentence (sum of log bigram probs)."""
        log_prob = 0.0
        for w_prev, w_curr in zip(sentence_tokens[:-1], sentence_tokens[1:]):
            p = self.word_probability(w_prev, w_curr)
            if p == 0:
                return float('-inf')
            log_prob += math.log(p)
        return log_prob


# Instantiate both models
model_unsmoothed = BigramLanguageModel(training_sentences, smoothing=False)
model_smoothed   = BigramLanguageModel(training_sentences, smoothing=True)

print(f"Test sentence: {' '.join(test_sentence)}\n")

p_us  = model_unsmoothed.sentence_probability(test_sentence)
p_s   = model_smoothed.sentence_probability(test_sentence)
lp_us = model_unsmoothed.sentence_log_probability(test_sentence)
lp_s  = model_smoothed.sentence_log_probability(test_sentence)

print(f"{'Model':<30} {'Probability':<30} {'Log Probability'}")
print("-" * 75)
print(f"{'Unsmoothed Bigram':<30} {p_us:<30.10f} {lp_us:.6f}")
print(f"{'Smoothed Bigram (Laplace)':<30} {p_s:<30.15f} {lp_s:.6f}")

Test sentence: <s> I read a different book by Danielle </s>

Model                          Probability                    Log Probability
---------------------------------------------------------------------------
Unsmoothed Bigram              0.0370370370                   -3.295837
Smoothed Bigram (Laplace)      0.000001224407021              -13.613054


In [7]:
# Detailed breakdown per bigram for both models
print("Detailed Per-Bigram Breakdown:\n")
print(f"{'Bigram':<32} {'Unsmoothed P':<20} {'Smoothed P'}")
print("-" * 70)

for w_prev, w_curr in bigrams_test:
    p_us_w = model_unsmoothed.word_probability(w_prev, w_curr)
    p_s_w  = model_smoothed.word_probability(w_prev, w_curr)
    label  = f"P({w_curr}|{w_prev})"
    print(f"{label:<32} {p_us_w:<20.6f} {p_s_w:.8f}")

Detailed Per-Bigram Breakdown:

Bigram                           Unsmoothed P         Smoothed P
----------------------------------------------------------------------
P(I|<s>)                         0.333333             0.15384615
P(read|I)                        1.000000             0.18181818
P(a|read)                        1.000000             0.30769231
P(different|a)                   0.333333             0.15384615
P(book|different)                1.000000             0.18181818
P(by|book)                       0.333333             0.15384615
P(Danielle|by)                   1.000000             0.18181818
P(</s>|Danielle)                 1.000000             0.18181818


### Observations and Analysis

| | Unsmoothed | Smoothed (Laplace) |
|-|-----------|--------------------|
| **P(sentence)** | ≈ 0.037037 (1/27) | ≈ 1.224 × 10⁻⁶ |
| **log P(sentence)** | ≈ −3.296 | ≈ −13.612 |
| **Handles unseen bigrams?** | ❌ Returns 0 (zero-prob problem) | ✅ Always > 0 |
| **Probability mass** | Concentrated on seen bigrams | Redistributed across V² possible bigrams |

**Key Insights:**

1. **Unsmoothed model (P = 1/27 ≈ 0.037):** All bigrams in the test sentence appear in the training corpus, so the model assigns a non-zero probability. However, if *any* bigram were unseen (e.g., a word absent from training), the entire sentence probability collapses to **0** — the *zero-probability problem*. This makes the model fragile for real-world applications.

2. **Laplace-smoothed model (P ≈ 1.22 × 10⁻⁶):** By adding 1 to every bigram count and normalising by V, probability mass is redistributed from seen to unseen bigrams. The result is much smaller because the same total probability (1.0) is now spread across all V² possible bigrams. This guarantees **no zero probabilities**, making it robust to unseen n-grams.

3. **Limitation of Laplace smoothing:** Although it solves the zero-probability problem, it tends to over-smooth — allocating too much probability to unseen events. More sophisticated methods such as **Good-Turing**, **Kneser-Ney**, or **interpolated smoothing** are preferred for production language models.